# OSAI Project 1 — Colab v1 Baseline Reproduction

Pascal-VOC 21-class semantic segmentation, ResNet-50 + DeepLabV3+ (OS=16). main 브랜치 기준 v1 baseline 재현.

재현을 위해 다음 항목들을 수행해주셔야 합니다.

1. Drive에 폴더 생성: `MyDrive/osai/p1/submit/img/` (1000장 jpg)
2. WandB API key Colab Secret에 저장

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 설정 변수

In [ ]:
REPO_URL = "https://github.com/geniemo/osai.git"
BRANCH   = "main"
CONFIG   = "src/config/colab.yaml"
DRIVE    = "/content/drive/MyDrive/osai/p1"

import os
os.makedirs(f"{DRIVE}/checkpoints", exist_ok=True)
print(f"Repo: {REPO_URL} (branch={BRANCH})")
print(f"Config: {CONFIG}")
print(f"Drive: {DRIVE}")
assert os.path.exists(f"{DRIVE}/submit/img"), f"submit/img not found at {DRIVE}/submit/img"
n_imgs = len([f for f in os.listdir(f"{DRIVE}/submit/img") if f.endswith(".jpg")])
print(f"submit/img: {n_imgs} jpg files")

## 2. 저장소 clone

In [ ]:
%cd /content
!rm -rf osai
!git clone --branch {BRANCH} --depth 1 {REPO_URL} osai
%cd osai/p1

## 3. uv 설치 + 의존성 sync

In [ ]:
!pip install -q uv
!uv sync

## 4. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
!uv run python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

## 5. WandB 로그인

In [ ]:
from google.colab import userdata
import os; os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')

!uv run wandb login

## 6. Test images Colab 로컬로 복사

In [ ]:
!mkdir -p submit/img
!cp -r {DRIVE}/submit/img/. submit/img/
# Drive submit/img/ → Colab submit/img/ 복사 (PDF 컨벤션)
!ls submit/img | wc -l
!ls submit/img | head -3

## 7. 데이터 다운로드 (VOC + COCO)

In [ ]:
!uv run python -m src.data.download --voc-root data/voc --coco-root data/coco

## 8. COCO mask cache 사전 생성

95K instance mask를 PNG로 1회 생성

In [ ]:
!uv run python -m src.build_coco_masks --coco-root data/coco --num-workers 4

## 9. 학습 (Stage 1 + Stage 2 자동, ~10-12h)

ckpt가 Drive에 저장됨. 세션 끊기면 다시 시작 시 자동 resume.

In [ ]:
!uv run python -m src.train --config {CONFIG}

## 10. Validation mIoU 측정

최종 ckpt로 val mIoU + 21 class별 IoU 출력.

In [ ]:
!uv run python -m src.eval --config {CONFIG} --ckpt {DRIVE}/checkpoints/model.pth

## 11. TTA Validation mIoU (선택, 실제 inference 성능)

Multi-scale + hflip TTA로 측정.

In [ ]:
!uv run python -m src.eval_tta --config {CONFIG} --ckpt {DRIVE}/checkpoints/model.pth

## 12. 추론 — submit/img → submit/pred (TTA)

PDF 재현 컨벤션과 동일. ~5–10분 (T4).

In [ ]:
!mkdir -p submit/pred
!uv run python -m src.infer \
    --config {CONFIG} \
    --ckpt {DRIVE}/checkpoints/model.pth \
    --input submit/img \
    --output submit/pred

## 13. ONNX export (가중치 제거, 채점용 ONNX)

`model_structure.onnx` 생성 (~0.3MB, ≤10MB 채점 한도).

In [ ]:
!uv run python -m src.export_onnx \
    --config {CONFIG} \
    --ckpt {DRIVE}/checkpoints/model.pth \
    --out {DRIVE}/model_structure.onnx

## 14. FLOPs 측정

In [ ]:
!uv run python -m src.measure_flops --onnx {DRIVE}/model_structure.onnx

## 15. PNG zip 생성

In [ ]:
!uv run python -m src.package_submission \
    --pred submit/pred \
    --out {DRIVE}/submission_pred.zip

## 16. 결과물 확인

In [ ]:
!ls -la {DRIVE}/
!ls -la {DRIVE}/checkpoints/